# Merge all workers after the team finishes (run once)
### Only one teammate should run this cell. Successful records take precedence over errors for the same `doc_id`.

In [ ]:
def iter_valid_jsonl(paths):
    for path in sorted(paths):
        with path.open('r', encoding='utf-8') as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    print(f'Warning: skipped malformed line {line_number} in {path.name}')

merged_success_path = MERGED_DIR / 'success.jsonl'
merged_error_path = MERGED_DIR / 'errors.jsonl'
successful_ids = set()
with merged_success_path.open('w', encoding='utf-8') as output:
    for record in iter_valid_jsonl(RESULTS_DIR.glob('worker_*_success.jsonl')):
        doc_id = str(record['doc_id'])
        if doc_id in successful_ids:
            continue
        successful_ids.add(doc_id)
        output.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')

error_ids = set()
with merged_error_path.open('w', encoding='utf-8') as output:
    for record in iter_valid_jsonl(RESULTS_DIR.glob('worker_*_errors.jsonl')):
        doc_id = str(record['doc_id'])
        if doc_id in successful_ids or doc_id in error_ids:
            continue
        error_ids.add(doc_id)
        output.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')

print(f'Merged {len(successful_ids):,} successes into {merged_success_path}')
print(f'Merged {len(error_ids):,} unresolved errors into {merged_error_path}')
